# Step 01. CARLA 구조 및 세팅

> **참고**: NGV_E2E_Education (Hanyang Univ.) — author Seokhwan Jeong (shjeong00@hanyang.ac.kr)  
> CARLA 0.9.16  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Day 1·2에서 만든 Planning 알고리즘을 **실제 시뮬레이터(CARLA)** 위에서 검증하기 위한 첫 단계입니다.  
이 노트북은 CARLA의 동작 구조(Server-Client, World, Actor)와 설치·세팅을 다룹니다.

## 학습 목표

1. CARLA의 Server-Client 구조 이해
2. World / Actor / Blueprint 등 핵심 객체 파악
3. 동기(synchronous) 모드와 `world.tick()` 개념
4. 설치 및 서버 실행 방법

## 1. CARLA란?

### 이론 배경

**CARLA**는 자율주행 연구용 오픈소스 시뮬레이터입니다 (Unreal Engine 기반).

- **Server**: 시뮬레이션과 렌더링 담당 (`./CarlaUE4.sh` 로 실행)
- **Client**: Python API로 Server에 접속해 명령 (`carla.Client`)
- 둘은 **TCP(기본 2000번 포트)** 로 통신합니다.

Client는 **World** 객체를 통해 맵·날씨·차량(Actor)을 읽고 제어합니다.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# CARLA Client-Server 구조 다이어그램 (영어 라벨)
fig, ax = plt.subplots(figsize=(11, 5)); ax.axis('off')
def box(x, y, w, h, label, color, fs=10):
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.02,rounding_size=0.08', facecolor=color, edgecolor='white', lw=2, alpha=0.95))
    ax.text(x, y, label, ha='center', va='center', color='white', fontsize=fs, fontweight='bold')

box(2.2, 3.0, 3.0, 1.4, 'CARLA Server\n(CarlaUE4.sh)\nsimulation + render', '#c0392b', 9)
box(8.0, 4.2, 3.0, 1.0, 'Python Client\ncarla.Client', '#2980b9', 9)
box(8.0, 2.7, 3.0, 1.0, 'World\n(map, weather, actors)', '#27ae60', 9)
box(8.0, 1.3, 3.0, 0.9, 'Actors\n(vehicle, sensor, ...)', '#8e44ad', 9)
ax.annotate('', xy=(6.5, 4.2), xytext=(3.7, 3.3), arrowprops=dict(arrowstyle='<->', color='gray', lw=2))
ax.annotate('', xy=(8.0, 3.7), xytext=(8.0, 3.2), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(8.0, 2.2), xytext=(8.0, 1.75), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.text(5.1, 4.0, 'TCP :2000', fontsize=8, color='dimgray', ha='center')
ax.set_xlim(0, 11); ax.set_ylim(0.5, 5)
ax.set_title('CARLA Architecture: Server <-> Client', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('Client가 TCP(기본 2000 포트)로 Server에 연결 -> World를 통해 Actor를 제어')


## 2. 핵심 객체

### 이론 배경

| 객체 | 설명 |
|------|------|
| `Client` | 서버 연결, 맵 로드 |
| `World` | 현재 시뮬레이션 (맵·날씨·액터 관리) |
| `Blueprint Library` | 스폰 가능한 차량·센서의 설계도 |
| `Actor` | 실제 스폰된 객체 (vehicle, sensor, walker) |
| `Transform` | 위치(`Location`) + 자세(`Rotation`) |
| `Spectator` | 관전 카메라 (시점 이동) |

좌표: CARLA는 **왼손 좌표계**(x 전방, y 우측, z 상방), 단위는 미터/도.

## 3. 동기 모드 (Synchronous Mode)

### 이론 배경

- **비동기**: 서버가 자체 속도로 진행 (실시간, 재현성 낮음)
- **동기**: 클라이언트가 `world.tick()` 을 호출해야 한 스텝 진행

```python
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05   # 한 틱 = 0.05초 (20Hz)
world.apply_settings(settings)
while True: world.tick()
```

Planning/Decision 실습은 **재현성**이 중요하므로 동기 모드를 사용합니다.

## 4. 설치 및 서버 실행

### 이론 배경

```bash
# 1) CARLA 0.9.16 서버 실행
cd ~/CARLA_0.9.16
./CarlaUE4.sh                 # (저사양: ./CarlaUE4.sh -quality-level=Low)

# 2) Python 패키지
pip install carla==0.9.16 pygame numpy
```

아래 셀로 `carla` 패키지 설치를 확인합니다 (서버가 꺼져 있어도 import는 됩니다).

In [ ]:
# carla 패키지 설치 확인 (서버 없이도 import는 가능)
try:
    import carla
    print(f'carla 패키지 OK — version {carla.__version__ if hasattr(carla, "__version__") else "0.9.16"}')
except ImportError:
    print('carla 패키지가 없습니다. CARLA 0.9.16의 PythonAPI를 설치하세요:')
    print('  pip install carla==0.9.16   (또는 PythonAPI/carla/dist 의 whl 설치)')


## 5. Day 3 커리큘럼

```
Step 01  CARLA 구조 및 세팅      <- 지금
Step 02  기본 실습 1            서버 연결, world 정보, 맵/날씨 변경
Step 03  기본 실습 2            ego 차량 spawn, spectator, spawn point
Step 04  Traffic Manager       다수 차량 + autopilot + TM 세팅
Step 05  판단 시나리오 1        cut-in / 선행차 감속 시나리오 제작
Step 06  판단 시나리오 2        TTC 기반 Decision (Day1/2 Planning 연계)
Step 07  AI 기반 Planning 개요   RL / IL / E2E 트렌드 (이론)
```